## **1 - MỤC TIÊU TUẦN 40**

Tuần 40 bắt đầu phần đóng góp riêng của khóa luận: xây **Attention-Gated Fusion** cho Composed Image Retrieval trên FashionIQ.

Baseline cũ:
`Q_add = normalize(I + T)`

Module mới:
`alpha = sigmoid(MLP([I; T]))`
`F = alpha * I + (1 - alpha) * T`

Trong đó `I` là FashionCLIP image embedding của candidate, `T` là FashionCLIP text embedding của modifier.

Notebook này nạp FashionCLIP embeddings và ánh xạ trực tiếp với tập train/val triplets. Sau bước kiểm tra forward pass, gate và gradient, hệ thống tiến hành train fusion module bằng contrastive loss. Kết quả cuối cùng được đo lường bằng Recall@10, Recall@50 và so sánh trực tiếp với additive baseline.

FashionCLIP được giữ cố định; chỉ fusion module được train.


## **2 - KIỂM TRA MÔI TRƯỜNG LOCAL**


In [1]:
import sys, platform, json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

print("Python ", sys.version.split()[0])
print("Platform ", platform.platform())
print("PyTorch ", torch.__version__)
print("CUDA available ", torch.cuda.is_available())

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device ", DEVICE)

Python  3.11.9
Platform  Windows-10-10.0.26200-SP0
PyTorch  2.13.0+cpu
CUDA available  False
Device  cpu


## **3 - CONFIG ĐƯỜNG DẪN**

Nếu embeddings nằm chỗ khác thì chỉ sửa `EMBEDDINGS_DIR`


In [2]:
PROJECT_ROOT = Path(r"C:\Users\Nguyen Ho Vinh  Hien\khoaluantotnghiep")
FASHIONIQ_ROOT = Path(r"C:\Users\Nguyen Ho Vinh  Hien\Downloads\FashionIQ\fashionIQ_dataset")
EMBEDDINGS_DIR = Path(r"C:\Users\Nguyen Ho Vinh  Hien\Downloads\KHOALUAN\Week37\embeddings")

TRAIN_CSV = PROJECT_ROOT / "data" / "processed" / "fashioniq_triplets_train.csv"
VAL_CSV = PROJECT_ROOT / "data" / "processed" / "fashioniq_triplets_val.csv"

IMAGE_EMB_PATH = EMBEDDINGS_DIR / "fashionclip_image_embeddings.npy"
IMAGE_IDS_PATH = EMBEDDINGS_DIR / "fashionclip_image_ids.csv"
TEXT_EMB_PATH = EMBEDDINGS_DIR / "fashionclip_text_embeddings.npy"
TEXT_META_PATH = EMBEDDINGS_DIR / "fashionclip_text_metadata.csv"

OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "week40_attention_fusion"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

required_paths = [
    TRAIN_CSV, VAL_CSV, IMAGE_EMB_PATH, IMAGE_IDS_PATH,
    TEXT_EMB_PATH, TEXT_META_PATH, FASHIONIQ_ROOT / "image_splits"
]

for p in required_paths:
    print(p, "->", p.exists())

if not all(p.exists() for p in required_paths):
    raise FileNotFoundError("Có đường dẫn chưa đung sửa Cell 3 trước khi chạy tiếp")

print("\nINPUT PATH CHECK PASS")


C:\Users\Nguyen Ho Vinh  Hien\khoaluantotnghiep\data\processed\fashioniq_triplets_train.csv -> True
C:\Users\Nguyen Ho Vinh  Hien\khoaluantotnghiep\data\processed\fashioniq_triplets_val.csv -> True
C:\Users\Nguyen Ho Vinh  Hien\Downloads\KHOALUAN\Week37\embeddings\fashionclip_image_embeddings.npy -> True
C:\Users\Nguyen Ho Vinh  Hien\Downloads\KHOALUAN\Week37\embeddings\fashionclip_image_ids.csv -> True
C:\Users\Nguyen Ho Vinh  Hien\Downloads\KHOALUAN\Week37\embeddings\fashionclip_text_embeddings.npy -> True
C:\Users\Nguyen Ho Vinh  Hien\Downloads\KHOALUAN\Week37\embeddings\fashionclip_text_metadata.csv -> True
C:\Users\Nguyen Ho Vinh  Hien\Downloads\FashionIQ\fashionIQ_dataset\image_splits -> True

INPUT PATH CHECK PASS


## **4 - NẠP EMBEDDINGS VÀ METADATA**

FashionCLIP image/text embeddings đều 512 chiều. Text metadata chứa thông tin candidate, target, category, split và source file.


In [3]:
image_embeddings = np.load(IMAGE_EMB_PATH, mmap_mode="r")
text_embeddings = np.load(TEXT_EMB_PATH, mmap_mode="r")
image_ids_df = pd.read_csv(IMAGE_IDS_PATH)
text_meta = pd.read_csv(TEXT_META_PATH)

print("Image embeddings ", image_embeddings.shape)
print("Text embeddings ", text_embeddings.shape)
print("Image mapping rows ", len(image_ids_df))
print("Text metadata rows ", len(text_meta))
print("Text metadata columns ", text_meta.columns.tolist())

assert image_embeddings.shape[1] == 512
assert text_embeddings.shape[1] == 512
assert len(image_ids_df) == image_embeddings.shape[0]
assert len(text_meta) == text_embeddings.shape[0]

print("\nEMBEDDING LOAD PASS")

Image embeddings  (74381, 512)
Text embeddings  (90402, 512)
Image mapping rows  74381
Text metadata rows  90402
Text metadata columns  ['triplet_id', 'source_file', 'category', 'split', 'candidate', 'target', 'caption_1', 'caption_2', 'text']

EMBEDDING LOAD PASS


## **5 - TẠO MAPPING IMAGE ID VÀ TEXT ROW**

Image mapping được tự nhận diện cột ID. Text metadata chỉ giữ category-specific train/val files để tránh duplicate từ annotation tổng hợp.


In [4]:
preferred_cols = ["image_id", "id", "image", "name"]
image_id_col = next((c for c in preferred_cols if c in image_ids_df.columns), image_ids_df.columns[0])

image_ids_df[image_id_col] = image_ids_df[image_id_col].astype(str)
image_id_to_idx = {x: i for i, x in enumerate(image_ids_df[image_id_col].tolist())}

required_text_cols = {"source_file","category","split","candidate","target","text"}
missing = required_text_cols - set(text_meta.columns)
if missing:
    raise ValueError(f"Text metadata thiếu cột  {sorted(missing)}")

text_meta = text_meta.copy()
text_meta["text_emb_idx"] = np.arange(len(text_meta))

allowed_sources = {
    f"cap.{category}.{split}.json"
    for category in ["dress","shirt","toptee"]
    for split in ["train","val"]
}

text_meta_clean = text_meta[text_meta["source_file"].isin(allowed_sources)].copy()

for c in ["candidate","target","category","split"]:
    text_meta_clean[c] = text_meta_clean[c].astype(str)

text_meta_clean = text_meta_clean.drop_duplicates(
    subset=["candidate","target","category","split"],
    keep="first"
)

print("Image ID column ", image_id_col)
print("Unique image IDs ", len(image_id_to_idx))
print("\nText rows by split/category ")
print(text_meta_clean.groupby(["split","category"]).size())

print("\nMAPPING PREP PASS")

Image ID column  image_id
Unique image IDs  74381

Text rows by split/category 
split  category
train  dress       5985
       shirt       5988
       toptee      6027
val    dress       2017
       shirt       2038
       toptee      1961
dtype: int64

MAPPING PREP PASS


## **6 - ÁNH XẠ TRIPLET CSV SANG EMBEDDING ROW**

Map bằng `candidate + target + category + split`, không phụ thuộc khác biệt dấu câu của modifier.


In [5]:
train_df = pd.read_csv(TRAIN_CSV, dtype=str)
val_df = pd.read_csv(VAL_CSV, dtype=str)

def build_index(df, split):
    x = df.copy()
    x["split"] = split

    meta = text_meta_clean[text_meta_clean["split"].eq(split)][
        ["candidate","target","category","split","text_emb_idx","text"]
    ]

    merged = x.merge(
        meta,
        on=["candidate","target","category","split"],
        how="left",
        validate="many_to_one"
    )

    merged["candidate_emb_idx"] = merged["candidate"].map(image_id_to_idx)
    merged["target_emb_idx"] = merged["target"].map(image_id_to_idx)

    print(f"\n{split.upper()}")
    print("Rows ", len(merged))
    print("Missing candidate ", merged["candidate_emb_idx"].isna().sum())
    print("Missing target ", merged["target_emb_idx"].isna().sum())
    print("Missing text ", merged["text_emb_idx"].isna().sum())

    valid = merged.dropna(
        subset=["candidate_emb_idx","target_emb_idx","text_emb_idx"]
    ).copy()

    for c in ["candidate_emb_idx","target_emb_idx","text_emb_idx"]:
        valid[c] = valid[c].astype(int)

    print("Valid rows ", len(valid))
    return valid.reset_index(drop=True)

train_index = build_index(train_df, "train")
val_index = build_index(val_df, "val")

assert len(train_index) > 0 and len(val_index) > 0
print("\nTRIPLET -> EMBEDDING MAPPING PASS")


TRAIN
Rows  18000
Missing candidate  0
Missing target  0
Missing text  0
Valid rows  18000

VAL
Rows  6016
Missing candidate  0
Missing target  0
Missing text  0
Valid rows  6016

TRIPLET -> EMBEDDING MAPPING PASS


## **7 - TẠO TENSOR CHO FUSION**

Mỗi sample chỉ còn 3 vector 512 chiều: candidate image, modifier text, target image.


In [6]:
def gather_tensor(arr, idx):
    return torch.from_numpy(
        np.asarray(arr[idx], dtype=np.float32).copy()
    )

train_image = gather_tensor(image_embeddings, train_index["candidate_emb_idx"].to_numpy())
train_text = gather_tensor(text_embeddings, train_index["text_emb_idx"].to_numpy())
train_target = gather_tensor(image_embeddings, train_index["target_emb_idx"].to_numpy())

val_image = gather_tensor(image_embeddings, val_index["candidate_emb_idx"].to_numpy())
val_text = gather_tensor(text_embeddings, val_index["text_emb_idx"].to_numpy())
val_target = gather_tensor(image_embeddings, val_index["target_emb_idx"].to_numpy())

print("Train ", train_image.shape, train_text.shape, train_target.shape)
print("Val ", val_image.shape, val_text.shape, val_target.shape)

assert train_image.shape == train_text.shape == train_target.shape
assert val_image.shape == val_text.shape == val_target.shape
assert train_image.shape[1] == 512

print("\nFUSION TENSOR DATASET PASS")

Train  torch.Size([18000, 512]) torch.Size([18000, 512]) torch.Size([18000, 512])
Val  torch.Size([6016, 512]) torch.Size([6016, 512]) torch.Size([6016, 512])

FUSION TENSOR DATASET PASS


## **8 - NẠP ATTENTION-GATED FUSION MODULE**


In [7]:
import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.attention_fusion import AttentionGatedFusion, in_batch_contrastive_loss

model = AttentionGatedFusion(
    embedding_dim=512,
    hidden_dim=512,
    dropout=0.1,
).to(DEVICE)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(model)
print("\nTrainable parameters ", f"{trainable_params:,}")

AttentionGatedFusion(
  (gate_network): Sequential(
    (0): Linear(in_features=1024, out_features=512, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=512, out_features=512, bias=True)
    (4): Sigmoid()
  )
  (projection): Sequential(
    (0): Linear(in_features=512, out_features=512, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=512, out_features=512, bias=True)
  )
)

Trainable parameters  1,312,768


## **9 - FORWARD PASS VÀ GATE SANITY CHECK**

Kiểm tra output `[B,512]`, gate `[B,512]`, gate nằm trong [0,1], query đã L2-normalized.


In [8]:
model.eval()

sample_image = train_image[:8].to(DEVICE)
sample_text = train_text[:8].to(DEVICE)

with torch.no_grad():
    sample_query, sample_gate = model(sample_image, sample_text, return_gate=True)

print("Image ", sample_image.shape)
print("Text ", sample_text.shape)
print("Query ", sample_query.shape)
print("Gate ", sample_gate.shape)
print("Gate min/max/mean ",
      float(sample_gate.min()),
      float(sample_gate.max()),
      float(sample_gate.mean()))

norms = sample_query.norm(dim=-1)
print("Query norms ", norms.cpu())

assert sample_query.shape == (8,512)
assert sample_gate.shape == (8,512)
assert sample_gate.min() >= 0 and sample_gate.max() <= 1
assert torch.allclose(norms, torch.ones_like(norms), atol=1e-4)

print("\nFORWARD PASS PASS")


Image  torch.Size([8, 512])
Text  torch.Size([8, 512])
Query  torch.Size([8, 512])
Gate  torch.Size([8, 512])
Gate min/max/mean  0.4813153147697449 0.5179797410964966 0.5004419684410095
Query norms  tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])

FORWARD PASS PASS


## **10 - GRADIENT SANITY CHECK**

Chứng minh module thật sự learnable bằng một bước loss + backward.


In [9]:
model.train()
model.zero_grad()

sample_target = train_target[:8].to(DEVICE)
query = model(sample_image, sample_text)
loss, _ = in_batch_contrastive_loss(query, sample_target, temperature=0.07)
loss.backward()

grad_params = sum(1 for p in model.parameters() if p.grad is not None)
grad_norm = sum(float(p.grad.norm()) for p in model.parameters() if p.grad is not None)

print("Loss ", loss.item())
print("Parameters with gradient ", grad_params)
print("Sum gradient norm ", grad_norm)

assert grad_params > 0 and grad_norm > 0
print("\nGRADIENT CHECK PASS")

Loss  1.9956797361373901
Parameters with gradient  8
Sum gradient norm  5.703796835616231

GRADIENT CHECK PASS


C:\Users\Nguyen Ho Vinh  Hien\AppData\Local\Temp\ipykernel_22208\1180039425.py:12: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:823.)
  print("Loss ", float(loss))


## **11 - TRAIN ATTENTION-GATED FUSION**

Mặc định 15 epoch, batch size 256, AdamW. Nếu muốn test nhanh trước, đổi `EPOCHS = 5`.


In [10]:
from torch.utils.data import TensorDataset, DataLoader
from torch.optim import AdamW

SEED = 42
EPOCHS = 15
BATCH_SIZE = 256
LEARNING_RATE = 1e-3
TEMPERATURE = 0.07

torch.manual_seed(SEED)
np.random.seed(SEED)

dataset = TensorDataset(train_image, train_text, train_target)
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    drop_last=True,
)

model = AttentionGatedFusion(512, 512, 0.1).to(DEVICE)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, total_items = 0.0, 0

    for ib, tb, gb in loader:
        ib, tb, gb = ib.to(DEVICE), tb.to(DEVICE), gb.to(DEVICE)

        optimizer.zero_grad()
        qb = model(ib, tb)
        loss, _ = in_batch_contrastive_loss(qb, gb, TEMPERATURE)
        loss.backward()
        optimizer.step()

        n = ib.size(0)
        total_loss += loss.item() * n
        total_items += n

    epoch_loss = total_loss / total_items
    history.append({"epoch": epoch, "train_loss": epoch_loss})
    print(f"Epoch {epoch:02d}/{EPOCHS} | train_loss = {epoch_loss:.6f}")

history_df = pd.DataFrame(history)
display(history_df)

history_df.to_csv(OUTPUT_DIR / "training_history.csv", index=False)
torch.save(model.state_dict(), OUTPUT_DIR / "attention_gated_fusion.pt")

print("\nTRAINING COMPLETE")

Epoch 01/15 | train_loss = 2.624924
Epoch 02/15 | train_loss = 2.295393
Epoch 03/15 | train_loss = 2.183521
Epoch 04/15 | train_loss = 2.094457
Epoch 05/15 | train_loss = 2.029626
Epoch 06/15 | train_loss = 1.968689
Epoch 07/15 | train_loss = 1.913794
Epoch 08/15 | train_loss = 1.861262
Epoch 09/15 | train_loss = 1.814381
Epoch 10/15 | train_loss = 1.767792
Epoch 11/15 | train_loss = 1.725570
Epoch 12/15 | train_loss = 1.678723
Epoch 13/15 | train_loss = 1.646982
Epoch 14/15 | train_loss = 1.607794
Epoch 15/15 | train_loss = 1.572023


,epoch,train_loss
0,1,2.624924
1,2,2.295393
2,3,2.183521
3,4,2.094457
4,5,2.029626
5,6,1.968689
6,7,1.913794
7,8,1.861262
8,9,1.814381
9,10,1.767792



TRAINING COMPLETE


## **12 - TẠO VALIDATION QUERIES**

Tạo cả additive baseline và attention-gated queries trên cùng validation set.


In [11]:
model.eval()

with torch.no_grad():
    additive_queries = F.normalize(val_image + val_text, dim=-1)

    parts = []
    eval_batch = 512

    for start in range(0, len(val_image), eval_batch):
        q = model(
            val_image[start:start+eval_batch].to(DEVICE),
            val_text[start:start+eval_batch].to(DEVICE),
        ).cpu()
        parts.append(q)

    attention_queries = torch.cat(parts, dim=0)

print("Additive ", additive_queries.shape)
print("Attention ", attention_queries.shape)

assert additive_queries.shape == attention_queries.shape
print("\nVALIDATION QUERY BUILD PASS")

Additive  torch.Size([6016, 512])
Attention  torch.Size([6016, 512])

VALIDATION QUERY BUILD PASS


## **13 - ĐÁNH GIÁ RECALL@10 / RECALL@50**

Giữ cùng FashionIQ validation protocol: gallery theo category, loại candidate khỏi ranking, cosine similarity, target là ground truth.


In [12]:
def evaluate_queries(query_embeddings, index_df, method_name, ks=(10,50), batch_size=256):
    results = []
    query_embeddings = F.normalize(query_embeddings.float(), dim=-1)

    for category in ["dress","shirt","toptee"]:
        mask = index_df["category"].eq(category).to_numpy()
        rows = index_df[mask].reset_index(drop=True)
        qcat = query_embeddings[torch.from_numpy(np.where(mask)[0])]

        split_path = FASHIONIQ_ROOT / "image_splits" / f"split.{category}.val.json"
        with open(split_path, "r", encoding="utf-8") as f:
            gallery_ids = [str(x) for x in json.load(f)]

        gallery_ids = [x for x in gallery_ids if x in image_id_to_idx]
        gallery_idx = np.array([image_id_to_idx[x] for x in gallery_ids], dtype=np.int64)

        gallery = torch.from_numpy(
            np.asarray(image_embeddings[gallery_idx], dtype=np.float32).copy()
        )
        gallery = F.normalize(gallery, dim=-1)

        local = {x:i for i,x in enumerate(gallery_ids)}
        targets = np.array([local.get(str(x), -1) for x in rows["target"]], dtype=np.int64)
        candidates = np.array([local.get(str(x), -1) for x in rows["candidate"]], dtype=np.int64)

        valid = targets >= 0
        qcat = qcat[torch.from_numpy(np.where(valid)[0])]
        targets = targets[valid]
        candidates = candidates[valid]

        hits = {k:0 for k in ks}
        total = 0

        for start in range(0, len(qcat), batch_size):
            qb = qcat[start:start+batch_size]
            sims = qb @ gallery.T

            cb = candidates[start:start+len(qb)]
            tb = targets[start:start+len(qb)]

            for i, cand in enumerate(cb):
                if cand >= 0:
                    sims[i, cand] = -1e9

            top = sims.topk(min(max(ks), sims.shape[1]), dim=1).indices.numpy()

            for k in ks:
                hits[k] += sum(t in top[i,:k] for i,t in enumerate(tb))

            total += len(tb)

        results.append({
            "method": method_name,
            "category": category,
            "queries": total,
            "recall@10": hits[10]/total,
            "recall@50": hits[50]/total,
        })

    df = pd.DataFrame(results)
    total_q = df["queries"].sum()

    macro = {
        "method": method_name,
        "category": "Macro Average",
        "queries": total_q,
        "recall@10": df["recall@10"].mean(),
        "recall@50": df["recall@50"].mean(),
    }

    micro = {
        "method": method_name,
        "category": "Micro Overall",
        "queries": total_q,
        "recall@10": (df["recall@10"]*df["queries"]).sum()/total_q,
        "recall@50": (df["recall@50"]*df["queries"]).sum()/total_q,
    }

    return pd.concat([df, pd.DataFrame([macro,micro])], ignore_index=True)

additive_results = evaluate_queries(additive_queries, val_index, "Additive")
attention_results = evaluate_queries(attention_queries, val_index, "Attention-Gated")

comparison = pd.concat([additive_results, attention_results], ignore_index=True)
comparison["R@10 (%)"] = (comparison["recall@10"]*100).round(2)
comparison["R@50 (%)"] = (comparison["recall@50"]*100).round(2)

display(comparison[["method","category","queries","R@10 (%)","R@50 (%)"]])

comparison.to_csv(OUTPUT_DIR / "attention_vs_additive_recall.csv", index=False)
print("\nRECALL EVALUATION COMPLETE")

,method,category,queries,R@10 (%),R@50 (%)
0,Additive,dress,2017,21.96,41.79
1,Additive,shirt,2038,22.13,37.63
2,Additive,toptee,1961,26.82,45.18
3,Additive,Macro Average,6016,23.64,41.54
4,Additive,Micro Overall,6016,23.60,41.49
5,Attention-Gated,dress,2017,36.09,59.59
6,Attention-Gated,shirt,2038,37.98,57.07
7,Attention-Gated,toptee,1961,42.17,65.88
8,Attention-Gated,Macro Average,6016,38.75,60.85
9,Attention-Gated,Micro Overall,6016,38.71,60.79



RECALL EVALUATION COMPLETE


## **14 - BẢNG SO SÁNH CUỐI**


In [13]:
final_table = comparison[
    comparison["category"].eq("Micro Overall")
][["method","queries","R@10 (%)","R@50 (%)"]].reset_index(drop=True)

display(final_table)

add_row = final_table[final_table["method"].eq("Additive")].iloc[0]
att_row = final_table[final_table["method"].eq("Attention-Gated")].iloc[0]

print("Delta R@10:",
      f"{att_row['R@10 (%)'] - add_row['R@10 (%)']:+.2f} percentage points")
print("Delta R@50:",
      f"{att_row['R@50 (%)'] - add_row['R@50 (%)']:+.2f} percentage points")

print("\nWEEK 40 ATTENTION-FUSION COMPLETE")

,method,queries,R@10 (%),R@50 (%)
0,Additive,6016,23.60,41.49
1,Attention-Gated,6016,38.71,60.79


Delta R@10: +15.11 percentage points
Delta R@50: +19.30 percentage points

WEEK 40 ATTENTION-FUSION COMPLETE


## **15 - PROTOCOL / DATA LEAKAGE SANITY CHECK**

Cell này kiểm tra các điểm quan trọng trước khi dùng kết quả Attention-Gated làm số liệu báo cáo:

1. Train và validation không chứa **exact triplet trùng nhau**.
2. Validation vẫn có đúng **6.016 query**, với đúng số lượng theo từng category.
3. Query của Attention-Gated chỉ được tạo từ `candidate image embedding + modifier text embedding`; `target embedding` chỉ được dùng trong loss/evaluation.
4. So sánh additive baseline hiện tại với kết quả Week 38 đã báo cáo: **R@10 = 23.64%, R@50 = 41.49%**.

Lưu ý: việc một image ID xuất hiện ở cả train và val không tự động là leakage; FashionIQ có thể tái sử dụng ảnh trong nhiều triplet. Kiểm tra quan trọng ở đây là exact triplet và luồng tạo query.


In [15]:
triplet_cols = ["candidate", "modifier", "target", "category"]
train_triplets = set(map(tuple, train_df[triplet_cols].fillna("").astype(str).to_numpy()))
val_triplets = set(map(tuple, val_df[triplet_cols].fillna("").astype(str).to_numpy()))
exact_overlap = train_triplets.intersection(val_triplets)

print("1 - EXACT TRIPLET OVERLAP")
print("Train unique triplets ", len(train_triplets))
print("Val unique triplets ", len(val_triplets))
print("Exact overlap ", len(exact_overlap))
print("PASS - Không có exact triplet trùng giữa train và val." if len(exact_overlap)==0 else "WARNING - Có exact triplet trùng giữa train và val.")

expected_val_counts = {"dress": 2017, "shirt": 2038, "toptee": 1961}
actual_val_counts = (val_index["category"].value_counts().reindex(["dress","shirt","toptee"]).fillna(0).astype(int).to_dict())
counts_ok = actual_val_counts == expected_val_counts and len(val_index) == 6016
print("\n2 - VALIDATION QUERY COUNTS")
print("Expected ", expected_val_counts)
print("Actual ", actual_val_counts)
print("Total ", len(val_index))
print("PASS" if counts_ok else "WARNING")

model.eval()
with torch.no_grad():
    q_before = model(val_image[:32].to(DEVICE), val_text[:32].to(DEVICE)).cpu()
    shuffled_target = val_target[:32][torch.randperm(32)].clone()
    q_after = model(val_image[:32].to(DEVICE), val_text[:32].to(DEVICE)).cpu()
query_target_independent = torch.allclose(q_before, q_after, atol=1e-7)
print("\n3 - QUERY DOES NOT USE TARGET AS INPUT")
print("Shuffled target shape ", shuffled_target.shape)
print("Queries unchanged ", query_target_independent)
print("PASS" if query_target_independent else "WARNING")

week38_reference = {"R@10 (%)": 23.64, "R@50 (%)": 41.49}
current_additive = final_table[final_table["method"].eq("Additive")].iloc[0]
current_r10 = float(current_additive["R@10 (%)"])
current_r50 = float(current_additive["R@50 (%)"])
delta_r10_week38 = current_r10 - week38_reference["R@10 (%)"]
delta_r50_week38 = current_r50 - week38_reference["R@50 (%)"]
print("\n4 - ADDITIVE BASELINE CONSISTENCY")
print(f"Week 38 reference - R@10={week38_reference['R@10 (%)']:.2f}% | R@50={week38_reference['R@50 (%)']:.2f}%")
print(f"Current notebook - R@10={current_r10:.2f}% | R@50={current_r50:.2f}%")
print(f"Difference - R@10={delta_r10_week38:+.2f} pp | R@50={delta_r50_week38:+.2f} pp")
if abs(delta_r10_week38) < 1e-9 and abs(delta_r50_week38) < 1e-9:
    print("PASS - Baseline khớp chính xác Week 38")
else:
    print("NOTE - Baseline không khớp tuyệt đối Week 38. R@50 khớp, R@10 chỉ lệch rất nhỏ")

core_checks_ok = len(exact_overlap)==0 and counts_ok and query_target_independent
print("\n5 - FINAL SANITY STATUS")
print("CORE PROTOCOL / DATA LEAKAGE CHECK PASS" if core_checks_ok else "CORE PROTOCOL / DATA LEAKAGE CHECK NEEDS REVIEW")

1 - EXACT TRIPLET OVERLAP
Train unique triplets  18000
Val unique triplets  6016
Exact overlap  0
PASS - Không có exact triplet trùng giữa train và val.

2 - VALIDATION QUERY COUNTS
Expected  {'dress': 2017, 'shirt': 2038, 'toptee': 1961}
Actual  {'dress': 2017, 'shirt': 2038, 'toptee': 1961}
Total  6016
PASS

3 - QUERY DOES NOT USE TARGET AS INPUT
Shuffled target shape  torch.Size([32, 512])
Queries unchanged  True
PASS

4 - ADDITIVE BASELINE CONSISTENCY
Week 38 reference - R@10=23.64% | R@50=41.49%
Current notebook - R@10=23.60% | R@50=41.49%
Difference - R@10=-0.04 pp | R@50=+0.00 pp
NOTE - Baseline không khớp tuyệt đối Week 38. R@50 khớp, R@10 chỉ lệch rất nhỏ

5 - FINAL SANITY STATUS
CORE PROTOCOL / DATA LEAKAGE CHECK PASS


## **16 - KẾT LUẬN VÀ HÌNH NÊN CHỤP**

Nếu Cell 15 trả về `CORE PROTOCOL / DATA LEAKAGE CHECK PASS` thì pipeline tuần 40 đã qua các kiểm tra cốt lõi về tách train/val, số lượng validation query và việc target không tham gia tạo query.

Kết quả Attention-Gated hiện tại nên được mô tả là **kết quả thực nghiệm ban đầu trên cùng evaluator của notebook tuần 40**. Vì additive R@10 hiện tại lệch rất nhỏ so với Week 38 (23.60% so với 23.64%), không nên gọi là “tái lập chính xác Week 38”; tuy nhiên Additive và Attention-Gated trong notebook này vẫn được đánh giá bằng cùng một evaluator nên phép so sánh nội bộ là nhất quán.

Các hình quan trọng để chụp:
1. Cell 6: Triplet -> Embedding Mapping PASS.
2. Cell 9 + 10: Forward Pass PASS và Gradient Check PASS.
3. Cell 11: training loss giảm theo epoch.
4. Cell 14: bảng Additive vs Attention-Gated.
5. Cell 15: `CORE PROTOCOL / DATA LEAKAGE CHECK PASS`.

Multi-head attention chưa triển khai ở tuần này. Nó chỉ nên được xem là bước nâng cấp sau khi gated-fusion đã được kiểm chứng và khi có token-level text features / patch-level image features phù hợp.
